# Native CLM v0 — M0 + M1

This notebook runs the first real token-predictive Native CLM pipeline.

- **M0:** architecture/runtime smoke, including dynamic Cell spawn and checkpoint round-trip.
- **M1:** canonical ~12M next-token training with one learned sparse Cellular Layer.
- **Publication:** lightweight results are committed back to `codex/native-clm-v0-m0-m1`.

M1 intentionally does **not** claim continual learning or autonomous growth. Those remain M2/M3.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

BRANCH = "codex/native-clm-v0-m0-m1"
REPO = Path("/kaggle/working/mini-cells")
DATA = Path("/kaggle/working/native-clm-m1-data")
M0_OUT = REPO / "artifacts/experiments/native-clm-v0-m0-execution-smoke"
M1_OUT = REPO / "artifacts/experiments/native-clm-v0-m1-next-token"

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)

if not (REPO / ".git").exists():
    run(["git", "clone", "--branch", BRANCH, "https://github.com/ArcheLabs/mini-cells.git", REPO])

os.chdir(REPO)
run(["git", "fetch", "origin"])
run(["git", "checkout", BRANCH])
run(["git", "pull", "--ff-only", "origin", BRANCH])
run([sys.executable, "-m", "pip", "install", "-e", ".[dev,lm]"])

import torch
print("branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Canonical M1 is a GPU run; enable a Kaggle GPU accelerator."


In [ ]:
# M0 — cheap execution/runtime gate.
run([
    sys.executable,
    "scripts/research/run_native_clm_v0_m0.py",
    "--output-dir",
    M0_OUT,
])
m0 = json.loads((M0_OUT / "decision.json").read_text())
print(json.dumps({"status": m0["status"], "gates": m0["gates"]}, indent=2))
assert m0["pass"] is True


In [ ]:
# Prepare the registered TinyStories cache.
run([
    sys.executable,
    "scripts/research/prepare_native_clm_v0_m1_data.py",
    "--dataset-id", "roneneldan/TinyStories",
    "--train-docs", "50000",
    "--validation-docs", "2000",
    "--output-dir", DATA,
])
manifest = json.loads((DATA / "manifest.json").read_text())
print(json.dumps(manifest, indent=2))


In [ ]:
# M1 — canonical ~12M next-token training.
# The runner returns code 2 when engineering gates are incomplete. We intentionally
# continue to publication so negative/incomplete results are preserved.
cmd = [
    sys.executable,
    "scripts/research/run_native_clm_v0_m1.py",
    "--config", "research/stages/06-native-clm/configs/native-clm-v0-m1-12m.json",
    "--train-file", DATA / "train.txt",
    "--validation-file", DATA / "validation.txt",
    "--output-dir", M1_OUT,
    "--device", "cuda",
]
print("+", " ".join(map(str, cmd)))
completed = subprocess.run(list(map(str, cmd)), check=False)
print("M1 runner return code:", completed.returncode)

shutil.copy2(DATA / "manifest.json", M1_OUT / "data-manifest.json")
summary = json.loads((M1_OUT / "summary.json").read_text())
print(json.dumps({
    "status": summary["status"],
    "pass": summary["pass"],
    "parameters": summary["parameter_count"]["total"],
    "initial_eval": summary["initial_eval"],
    "final_eval": summary["final_eval"],
    "cell_count": summary["cell_count"],
    "active_cells": summary["active_cells"],
    "checkpoint_sha256": summary["final_checkpoint_sha256"],
}, indent=2))
print("\nGates:")
for name, passed in summary["gates"].items():
    print(f"  {name}: {passed}")
print("\nGeneration sample:\n")
print((M1_OUT / "sample.txt").read_text(errors="replace"))


In [ ]:
# Publish exact lightweight M0/M1 results back to the research branch.
# Binary checkpoints stay in Kaggle; summary.json records their identity.
from kaggle_secrets import UserSecretsClient

os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
assert os.environ["GITHUB_TOKEN"], "Missing Kaggle Secret: GITHUB_TOKEN"

run([
    sys.executable,
    "scripts/research/publish_native_clm_v0_m1.py",
    "--branch", BRANCH,
])

print("\nPublished Native CLM v0 M0/M1 results.")
print("M1 status:", summary["status"])
if not summary["pass"]:
    print("M1 is incomplete under the registered engineering gates; the result was still published.")
